# AI Support Assistant — NLP Exploration Notebook

This notebook walks through every NLP concept used in the project:
1. Text Preprocessing Pipeline
2. Intent Classification (TF-IDF + ML)
3. Sentiment Analysis
4. Semantic Search with Embeddings
5. End-to-End Pipeline Demo

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

plt.style.use('dark_background')
print('Setup complete')

## 1. Load Dataset

In [ ]:
with open('../datasets/support_data.json') as f:
    data = json.load(f)

rows = []
for intent_obj in data['intents']:
    for ex in intent_obj['examples']:
        rows.append({'text': ex, 'intent': intent_obj['intent']})

df = pd.DataFrame(rows)
print(f'Total samples: {len(df)}')
df['intent'].value_counts().plot(kind='barh', color='#6c63ff', figsize=(8,4))
plt.title('Intent Distribution')
plt.tight_layout()
plt.show()

## 2. Preprocessing Pipeline Demo

In [ ]:
from backend.utils.preprocessor import clean_text, tokenize, remove_stopwords, lemmatize, preprocess

sample = 'My Payment FAILED!!! Money was deducted from my account'
print('Original  :', sample)
print('Cleaned   :', clean_text(sample))
print('Tokenized :', tokenize(clean_text(sample)))
print('No stops  :', remove_stopwords(tokenize(clean_text(sample))))
print('Lemmatized:', lemmatize(remove_stopwords(tokenize(clean_text(sample)))))
print('Final     :', preprocess(sample))

## 3. Train Intent Classifier

In [ ]:
from backend.training.train_intent import train
best_model = train()

## 4. Confusion Matrix

In [ ]:
from sklearn.model_selection import train_test_split
from backend.training.train_intent import load_data

X, y = load_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
y_pred = best_model.predict(X_test)

labels = sorted(set(y))
cm = confusion_matrix(y_test, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=[l.replace('_', ' ') for l in labels])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
plt.title('Intent Classifier — Confusion Matrix')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 5. Sentiment Analysis

In [ ]:
from backend.services.sentiment_service import analyze_sentiment

test_texts = [
    'My payment failed and I am very frustrated!',
    'Thank you, the issue was resolved quickly.',
    'I want to check my order status.',
    'This is the worst service I have ever used. Scam!',
]

for t in test_texts:
    r = analyze_sentiment(t)
    print(f"[{r['sentiment'].upper():8}] polarity={r['polarity']:+.2f}  urgent={r['is_urgent']}  →  {t[:60]}")

## 6. Semantic Search — Embedding Visualization

In [ ]:
from backend.services.semantic_service import semantic_search

queries = [
    'money deducted but recharge failed',
    'I forgot my login credentials',
    'package has not arrived yet',
    'app is broken',
]

for q in queries:
    r = semantic_search(q)
    print(f"Query  : {q}")
    print(f"Matched: {r['matched_text']}  (score={r['score']})")
    print(f"Answer : {r['response'][:80]}...\n")

## 7. Full Pipeline End-to-End

In [ ]:
from backend.services.response_engine import process_query

result = process_query('My payment failed but money got deducted')
for k, v in result.items():
    print(f"{k:25}: {v}")